# Cogni AP Tutor — semantic quality evaluation

This notebook compares all 756 tuned-model responses with the held-out teacher references using an OpenAI-compatible judge API. Use a **CPU runtime**; no GPU is needed. Results save after every example and safely resume. Do not use these judgments as training data until the final evaluation is frozen.


## 1. Install and mount Drive


In [ ]:
!pip -q install -U requests
from google.colab import drive
drive.mount('/content/drive')


## 2. Enter the judge API settings
The key stays in memory and is not written to Drive. Use the same OpenAI-compatible endpoint that worked for teacher labeling.


In [ ]:
import getpass

BASE_URL = input('OpenAI-compatible base URL (without /chat/completions): ').strip().rstrip('/')
API_KEY = getpass.getpass('Judge API key: ').strip()
JUDGE_MODEL = input('Judge model/deployment name: ').strip()
assert BASE_URL.startswith('http') and API_KEY and JUDGE_MODEL
print('Judge credentials loaded in memory only.')


## 3. Load and verify the frozen evaluation artifacts


In [ ]:
import json, re, time, random
from pathlib import Path

TEST_FILE = Path('/content/drive/MyDrive/cogni/production_v1/test.jsonl')
PREDICTIONS_FILE = Path('/content/drive/MyDrive/cogni/evaluation/ap_tutor_base_vs_tuned_v1/tuned_predictions.jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/cogni/evaluation/ap_tutor_semantic_judge_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JUDGMENTS_FILE = OUTPUT_DIR / 'semantic_judgments.jsonl'

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

test_rows = read_jsonl(TEST_FILE)
prediction_rows = read_jsonl(PREDICTIONS_FILE)
predictions = {str(row['example_id']): row for row in prediction_rows}
assert len(test_rows) == 756
assert len(predictions) == 756

def row_id(row):
    return str(row.get('example_id') or row.get('id') or '')

def message_content(row, role):
    for message in row.get('messages', []):
        if message.get('role') == role:
            return str(message.get('content', '')).strip()
    return ''

records = []
for row in test_rows:
    eid = row_id(row)
    records.append({
        'example_id': eid,
        'student_argument': message_content(row, 'user'),
        'teacher_reference': message_content(row, 'assistant'),
        'tuned_response': predictions[eid]['response'],
    })
assert all(r['example_id'] and r['student_argument'] and r['teacher_reference'] and r['tuned_response'] for r in records)
print('Ready to judge:', len(records))


## 4. Define the judge and strict output validation


In [ ]:
import requests

JUDGE_SYSTEM = '''You are a rigorous evaluator of an AP English Language logical-fallacy tutor. Compare the candidate with the student argument and held-out teacher reference. Judge semantic quality, not exact wording. The teacher reference is evidence, not infallible: use sound logical reasoning. Return exactly one JSON object and no prose.

Required JSON schema:
{
  "primary_fallacy_correct": true,
  "explanation_correct": true,
  "analogy_quality": 1,
  "transfer_quality": 1,
  "overall_quality": 1,
  "critical_error": false,
  "error_categories": [],
  "brief_reason": "one concise sentence"
}

Scores are integers 1-5. A critical error means the primary fallacy/no-fallacy judgment is materially wrong, the explanation contradicts the argument, the transfer item reveals its answer, or the response is unusable/repetitive. error_categories may contain: wrong_fallacy, wrong_explanation, weak_analogy, weak_transfer, answer_revealed, format_failure, repetition, other.'''

def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
    text = re.sub(r'\s*```$', '', text)
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == '{':
            try:
                value, _ = decoder.raw_decode(text[index:])
                if isinstance(value, dict):
                    return value
            except json.JSONDecodeError:
                pass
    raise ValueError('response_not_json_object')

def validate_judgment(value):
    required_bools = ['primary_fallacy_correct', 'explanation_correct', 'critical_error']
    for key in required_bools:
        if not isinstance(value.get(key), bool):
            raise ValueError(f'invalid_{key}')
    for key in ['analogy_quality', 'transfer_quality', 'overall_quality']:
        score = value.get(key)
        if not isinstance(score, int) or not 1 <= score <= 5:
            raise ValueError(f'invalid_{key}')
    if not isinstance(value.get('error_categories'), list):
        raise ValueError('invalid_error_categories')
    if not isinstance(value.get('brief_reason'), str) or not value['brief_reason'].strip():
        raise ValueError('invalid_brief_reason')
    return value

def judge_one(record):
    user_prompt = (
        'STUDENT ARGUMENT:\n' + record['student_argument'] +
        '\n\nHELD-OUT TEACHER REFERENCE:\n' + record['teacher_reference'] +
        '\n\nCANDIDATE TUNED RESPONSE:\n' + record['tuned_response']
    )
    payload = {
        'model': JUDGE_MODEL, 'max_tokens': 700,
        'messages': [
            {'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user', 'content': user_prompt},
        ],
    }
    response = requests.post(
        BASE_URL + '/chat/completions',
        headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
        json=payload, timeout=180,
    )
    if not response.ok:
        raise RuntimeError(f'Judge HTTP {response.status_code}: {response.text[:1000]}')
    content = response.json()['choices'][0]['message']['content']
    judgment = validate_judgment(extract_json(content))
    return {'example_id': record['example_id'], 'valid': True, 'judgment': judgment}


## 5. Run a 5-example pilot
Confirm five valid judgments before launching all 756 calls.


In [ ]:
pilot = []
for record in records[:5]:
    result = judge_one(record)
    pilot.append(result)
    print(result['example_id'], result['judgment'])
assert len(pilot) == 5 and all(item['valid'] for item in pilot)
print('Pilot passed. Continue to the full run.')


## 6. Run all 756 judgments with resume and retries
Start with 8 workers. If the provider rate-limits requests, change `MAX_WORKERS` to 4.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 8

def completed_ids():
    if not JUDGMENTS_FILE.exists():
        return set()
    return {str(row['example_id']) for row in read_jsonl(JUDGMENTS_FILE) if row.get('valid')}

def judge_with_retry(record, attempts=4):
    error = None
    for attempt in range(1, attempts + 1):
        try:
            result = judge_one(record)
            result['attempt'] = attempt
            return result
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
            if attempt < attempts:
                time.sleep((2 ** (attempt - 1)) + random.random())
    return {'example_id': record['example_id'], 'valid': False, 'error': error, 'attempt': attempts}

done_ids = completed_ids()
pending = [record for record in records if record['example_id'] not in done_ids]
print(f'Already valid: {len(done_ids)}/756 | Pending: {len(pending)}')
started = time.time()
valid_new = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(judge_with_retry, record): record for record in pending}
    for processed, future in enumerate(as_completed(futures), 1):
        result = future.result()
        # Open and close for every row so Drive synchronizes resumable progress.
        with JUDGMENTS_FILE.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
        valid_new += int(result.get('valid') is True)
        if processed % 25 == 0 or processed == len(pending):
            elapsed = time.time() - started
            rate = processed / elapsed if elapsed else 0
            eta_min = (len(pending) - processed) / rate / 60 if rate else 0
            print(f'{processed}/{len(pending)} processed | valid {valid_new} | ETA {eta_min:.1f} min')

print('Semantic judging pass complete:', JUDGMENTS_FILE)


## 7. Compile the semantic-quality summary


In [ ]:
from collections import Counter
from statistics import mean

all_attempts = read_jsonl(JUDGMENTS_FILE)
latest_valid = {}
for row in all_attempts:
    if row.get('valid'):
        latest_valid[str(row['example_id'])] = row

assert len(latest_valid) == 756, f'Only {len(latest_valid)}/756 valid judgments; rerun Section 6.'
judgments = [row['judgment'] for row in latest_valid.values()]
categories = Counter(category for j in judgments for category in j['error_categories'])
summary = {
    'judge_model': JUDGE_MODEL,
    'example_count': len(judgments),
    'primary_fallacy_accuracy': mean(j['primary_fallacy_correct'] for j in judgments),
    'explanation_accuracy': mean(j['explanation_correct'] for j in judgments),
    'mean_analogy_quality_1_to_5': mean(j['analogy_quality'] for j in judgments),
    'mean_transfer_quality_1_to_5': mean(j['transfer_quality'] for j in judgments),
    'mean_overall_quality_1_to_5': mean(j['overall_quality'] for j in judgments),
    'critical_error_rate': mean(j['critical_error'] for j in judgments),
    'error_categories': dict(categories.most_common()),
}
(OUTPUT_DIR / 'semantic_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Saved:', OUTPUT_DIR / 'semantic_summary.json')


## Stop here
Paste the final semantic summary into Codex. We will use it to complete Step 2: correct the two long-input generations and make the evidence-based retraining decision.
